# Part D: Mechanical Failure Analysis — Logistic Regression (Champion)

This notebook performs the Part D failure analysis for the selected champion model.  
The objective is to inspect the raw validation data where the model was confidently incorrect, explain the failure mechanically based on feature values, and propose targeted technical fixes.

## 1. Setup: Load Artifacts & Data

In this section, we import the required libraries, load the saved champion model artifact from the `.joblib` file, and load the prepared train/test datasets from the Data Preparation stage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported.")

In [ ]:
# Load champion model artifact
artifact = joblib.load("logistic_regression_baseline.joblib")
champion_pipeline = artifact["pipeline"]
champion_threshold = artifact["threshold"]
le = artifact["label_encoder"]

print("Loaded champion artifact successfully.")
print(f"Champion threshold: {champion_threshold}")
print(f"Pipeline: {champion_pipeline}")

In [ ]:
# Load prepared data
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train_raw = pd.read_csv("y_train.csv").values.ravel()
y_test_raw = pd.read_csv("y_test.csv").values.ravel()

y_train = le.transform(y_train_raw)
y_test = le.transform(y_test_raw)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Positive rate (train): {np.mean(y_train):.4f}")
print(f"Positive rate (test):  {np.mean(y_test):.4f}")

## 2. Shared Validation Helpers

Part D requires analysis on validation data rather than only on the final test set.  
To achieve this, we generate out-of-fold predicted probabilities using cross-validation so that each training instance is scored by a model that did not train on that specific instance.

This makes the failure analysis more reliable and avoids leakage.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

TARGET_NAMES = ["No Heart Disease (0)", "Heart Disease (1)"]

def classify_from_threshold(y_proba, threshold):
    return (y_proba >= threshold).astype(int)

def get_error_type(y_true, y_pred):
    if y_true == y_pred:
        return "Correct"
    elif y_true == 1 and y_pred == 0:
        return "False Negative"
    else:
        return "False Positive"

print("Validation helpers ready.")

## 3. Generate Out-of-Fold Validation Predictions

In this section, we produce out-of-fold validation probabilities for the training set using the loaded champion model.  
These probabilities are then converted into thresholded decisions using the saved champion threshold from Part C.

In [ ]:
oof_proba = cross_val_predict(
    champion_pipeline,
    X_train,
    y_train,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

oof_pred = classify_from_threshold(oof_proba, champion_threshold)

print("OOF validation predictions generated.")
print("\nValidation classification report:")
print(classification_report(y_train, oof_pred, target_names=TARGET_NAMES))

## 4. Build the Validation Failure Table

This table stores the raw validation rows together with:
- actual label
- predicted probability score
- thresholded prediction
- failure type
- prediction confidence
- distance from threshold

This structure makes it easier to identify the most confidently wrong cases.

In [ ]:
validation_results = X_train.copy()
validation_results["actual"] = y_train
validation_results["predicted_prob"] = oof_proba
validation_results["predicted_label"] = oof_pred

validation_results["error_type"] = [
    get_error_type(a, p) for a, p in zip(validation_results["actual"], validation_results["predicted_label"])
]

validation_results["prediction_confidence"] = np.where(
    validation_results["predicted_label"] == 1,
    validation_results["predicted_prob"],
    1 - validation_results["predicted_prob"]
)

validation_results["distance_from_threshold"] = np.abs(
    validation_results["predicted_prob"] - champion_threshold
)

print(validation_results["error_type"].value_counts())

## 5. Extract the Most Confident Failure Cases

For classification failure analysis, the most useful cases are:
- false negatives with high confidence
- false positives with high confidence

Because this project is framed as a medical triage task, false negatives are especially important, since they represent cases where an at-risk patient was assigned a low-risk score.

In [ ]:
TOP_K = 5  # change to 10 if needed

false_negatives = (
    validation_results[validation_results["error_type"] == "False Negative"]
    .sort_values("prediction_confidence", ascending=False)
    .head(TOP_K)
    .copy()
)

false_positives = (
    validation_results[validation_results["error_type"] == "False Positive"]
    .sort_values("prediction_confidence", ascending=False)
    .head(TOP_K)
    .copy()
)

print("=" * 80)
print("TOP FALSE NEGATIVES (most confidently missed positive cases)")
print("=" * 80)
display(false_negatives[[
    "actual", "predicted_label", "predicted_prob",
    "prediction_confidence", "distance_from_threshold"
]])

print("=" * 80)
print("TOP FALSE POSITIVES (most confidently over-flagged negative cases)")
print("=" * 80)
display(false_positives[[
    "actual", "predicted_label", "predicted_prob",
    "prediction_confidence", "distance_from_threshold"
]])

## 6. Fit the Champion Model on the Full Training Data for Interpretation

Out-of-fold predictions are used to identify the failures, but to analyse feature contributions in a stable way, we fit the saved champion pipeline on the full training data.

This gives us access to the trained logistic regression coefficients for interpretation.

In [ ]:
interpret_model = champion_pipeline
interpret_model.fit(X_train, y_train)

scaler = interpret_model.named_steps["scaler"]
logreg = interpret_model.named_steps["logreg"]

feature_names = X_train.columns.tolist()
coefficients = logreg.coef_[0]
intercept = logreg.intercept_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "AbsCoefficient": np.abs(coefficients)
}).sort_values("AbsCoefficient", ascending=False)

print("Top 20 features by absolute coefficient:")
display(coef_df.head(20))

## 7. Helper Functions for Row-Level Mechanical Explanation

The following helper functions compute row-level feature contributions using the logistic regression coefficients.  
For each case, we can identify:
- features pushing the prediction toward heart disease
- features pushing the prediction away from heart disease

This supports the mechanical explanation required in Part D.

In [ ]:
def get_row_contributions(model, X_row):
    scaler = model.named_steps["scaler"]
    logreg = model.named_steps["logreg"]

    row_scaled = scaler.transform(X_row)
    contrib = row_scaled[0] * logreg.coef_[0]

    contrib_df = pd.DataFrame({
        "Feature": X_row.columns,
        "RawValue": X_row.iloc[0].values,
        "Contribution": contrib
    })

    contrib_df["AbsContribution"] = contrib_df["Contribution"].abs()
    contrib_df = contrib_df.sort_values("AbsContribution", ascending=False)
    return contrib_df

def summarise_case(model, X_source, result_df, row_idx, top_n=5):
    X_row = X_source.loc[[row_idx]]
    row_meta = result_df.loc[row_idx, [
        "actual", "predicted_label", "predicted_prob",
        "error_type", "prediction_confidence"
    ]]

    contrib_df = get_row_contributions(model, X_row)

    strongest_positive = contrib_df.sort_values("Contribution", ascending=False).head(top_n)
    strongest_negative = contrib_df.sort_values("Contribution", ascending=True).head(top_n)

    return row_meta, contrib_df, strongest_positive, strongest_negative

print("Case explanation helpers ready.")

## 8. Inspect Individual Failure Cases

This section loops through the selected failure cases and displays:
- predicted probability score
- actual outcome
- failure type
- strongest positive contributors
- strongest negative contributors
- raw feature values

In [ ]:
selected_cases = pd.concat([false_negatives, false_positives], axis=0)

for idx in selected_cases.index:
    row_meta, contrib_df, strongest_positive, strongest_negative = summarise_case(
        interpret_model, X_train, validation_results, idx, top_n=5
    )

    print("\n" + "=" * 100)
    print(f"CASE INDEX: {idx}")
    print("=" * 100)
    print(row_meta.to_string())

    print("\nTop features PUSHING prediction toward Heart Disease:")
    display(strongest_positive[["Feature", "RawValue", "Contribution"]])

    print("\nTop features PUSHING prediction away from Heart Disease:")
    display(strongest_negative[["Feature", "RawValue", "Contribution"]])

    print("\nFull row values:")
    display(X_train.loc[[idx]])

## 9. Build a Formal Failure Analysis Table

To make reporting easier, this section converts the inspected cases into a compact summary table.

The table records:
- the failure type
- the predicted risk score
- the confidence level
- the strongest positive and negative mechanical drivers

In [ ]:
def build_failure_summary_table(model, X_source, result_df, indices, top_n=3):
    rows = []

    for idx in indices:
        row_meta, contrib_df, strongest_positive, strongest_negative = summarise_case(
            model, X_source, result_df, idx, top_n=top_n
        )

        pos_summary = "; ".join(
            [f"{r.Feature} ({r.Contribution:.3f})" for _, r in strongest_positive.iterrows()]
        )
        neg_summary = "; ".join(
            [f"{r.Feature} ({r.Contribution:.3f})" for _, r in strongest_negative.iterrows()]
        )

        rows.append({
            "Index": idx,
            "Error Type": row_meta["error_type"],
            "Actual": "Yes" if row_meta["actual"] == 1 else "No",
            "Predicted Decision": "Yes" if row_meta["predicted_label"] == 1 else "No",
            "Predicted Risk Score": round(row_meta["predicted_prob"], 4),
            "Confidence": round(row_meta["prediction_confidence"], 4),
            "Strongest Positive Drivers": pos_summary,
            "Strongest Negative Drivers": neg_summary
        })

    return pd.DataFrame(rows)

failure_summary_table = build_failure_summary_table(
    interpret_model,
    X_train,
    validation_results,
    selected_cases.index.tolist(),
    top_n=3
)

print("Formal failure analysis table:")
display(failure_summary_table)

## 10. Aggregate Recurring Failure Patterns

Beyond individual examples, it is useful to identify repeated feature patterns across multiple failure cases.

This section aggregates:
- recurring negative drivers in false negatives
- recurring positive drivers in false positives

These patterns help justify targeted technical fixes.

In [ ]:
def aggregate_top_contributors(model, X_source, indices, direction="negative", top_n=5):
    collected = []

    for idx in indices:
        X_row = X_source.loc[[idx]]
        contrib_df = get_row_contributions(model, X_row)

        if direction == "negative":
            chosen = contrib_df.sort_values("Contribution", ascending=True).head(top_n)
        else:
            chosen = contrib_df.sort_values("Contribution", ascending=False).head(top_n)

        collected.extend(chosen["Feature"].tolist())

    summary = pd.Series(collected).value_counts().rename_axis("Feature").reset_index(name="Count")
    return summary

fn_negative_patterns = aggregate_top_contributors(
    interpret_model, X_train, false_negatives.index.tolist(), direction="negative", top_n=5
)

fp_positive_patterns = aggregate_top_contributors(
    interpret_model, X_train, false_positives.index.tolist(), direction="positive", top_n=5
)

print("Recurring negative drivers in False Negatives:")
display(fn_negative_patterns.head(15))

print("Recurring positive drivers in False Positives:")
display(fp_positive_patterns.head(15))

## 11. Proposed Technical Fixes

Based on the observed failure patterns, this section lists targeted fixes that can be discussed in the report.

These fixes should connect directly to the mechanical causes identified in the failure analysis rather than being generic modelling suggestions.

In [ ]:
proposed_fixes = pd.DataFrame([
    {
        "Observed Failure Pattern": "False negatives occur when several weak low-risk signals collectively overpower a smaller number of strong risk indicators.",
        "Mechanical Interpretation": "The linear model sums independent effects and may underrepresent interaction effects between clinically related features.",
        "Targeted Technical Fix": "Add interaction features such as Age x SmokerStatus, Diabetes x BMI, or Stroke x PhysicalHealthDays."
    },
    {
        "Observed Failure Pattern": "False positives occur when one or two strong features dominate an otherwise low-risk profile.",
        "Mechanical Interpretation": "The model may be overreacting to sparse one-hot indicators or strongly weighted coefficients.",
        "Targeted Technical Fix": "Use stronger regularisation, merge rare categories, or review coefficient stability."
    },
    {
        "Observed Failure Pattern": "Many cases cluster close to the decision threshold and flip easily between classes.",
        "Mechanical Interpretation": "The score ranking may be reasonable, but the operating rule may not align perfectly with the deployment objective.",
        "Targeted Technical Fix": "Revisit threshold selection if recall needs to be prioritised more strongly."
    },
    {
        "Observed Failure Pattern": "Some missed cases appear to involve nonlinear risk combinations that logistic regression treats additively.",
        "Mechanical Interpretation": "The chosen model family may be too simple for some patient subgroups.",
        "Targeted Technical Fix": "Engineer nonlinear features or compare against a tree-based champion if justified by earlier experiments."
    }
])

print("Proposed technical fixes:")
display(proposed_fixes)

## 12. Optional Visualisation of Selected Failure Cases

This section provides a simple visual view of the selected failure cases and their predicted risk scores relative to the chosen threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    false_negatives.index,
    false_negatives["predicted_prob"],
    label="False Negatives",
    marker="x",
    s=100
)

ax.scatter(
    false_positives.index,
    false_positives["predicted_prob"],
    label="False Positives",
    marker="o",
    s=80
)

ax.axhline(champion_threshold, linestyle="--", linewidth=2, label=f"Threshold = {champion_threshold:.2f}")
ax.set_xlabel("Row Index")
ax.set_ylabel("Predicted Risk Score")
ax.set_title("Most Confident Validation Errors")
ax.legend()
plt.tight_layout()
plt.show()

## 13. Save Part D Outputs

The final tables can be exported for later use in slides or in the written report.

In [ ]:
failure_summary_table.to_csv("partD_failure_summary_table.csv", index=False)
proposed_fixes.to_csv("partD_proposed_fixes.csv", index=False)
false_negatives.to_csv("partD_top_false_negatives.csv", index=True)
false_positives.to_csv("partD_top_false_positives.csv", index=True)

print("Saved:")
print("  partD_failure_summary_table.csv")
print("  partD_proposed_fixes.csv")
print("  partD_top_false_negatives.csv")
print("  partD_top_false_positives.csv")